# 01. 데이터 읽기 및 기본 확인

첫 단계에서는 원본 데이터를 변경하거나 전처리 결과를 저장하지 않는다. 데이터 경로, 파일 구성, 컬럼, 샘플 품질만 확인한다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [ ]:
def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
DATA_ROOT = (PROJECT_ROOT / '../../data/stock_data').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
RESULTS_ROOT = (PROJECT_ROOT / '../../results').resolve()

for name, path in {
    'project': PROJECT_ROOT,
    'data': DATA_ROOT,
    'model': MODEL_ROOT,
    'results': RESULTS_ROOT,
}.items():
    print(f'{name:>7}: {path} | exists={path.exists()}')

assert DATA_ROOT.is_dir(), f'데이터 경로가 없습니다: {DATA_ROOT}'

## 파일과 세션 확인

In [ ]:
files = sorted(DATA_ROOT.glob('raw/session_*/*_enriched.csv'))
assert files, f'enriched CSV가 없습니다: {DATA_ROOT}'

inventory = pd.DataFrame({
    'session': [path.parent.name for path in files],
    'file_name': [path.name for path in files],
    'source_file': [str(path) for path in files],
})

print(f'세션 수: {inventory.session.nunique():,}')
print(f'파일 수: {len(inventory):,}')
display(inventory.groupby('session').size().rename('file_count').to_frame())
display(inventory.head())

## 전체 파일 스키마 확인

전체 파일은 헤더만 읽어 컬럼 구성이 같은지 확인한다.

In [ ]:
REQUIRED_COLUMNS = ['symbol', 'timestamp_utc', 'open', 'high', 'low', 'close']
schema_rows = []

for path in files:
    columns = pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns.tolist()
    schema_rows.append({
        'session': path.parent.name,
        'file_name': path.name,
        'column_count': len(columns),
        'missing_required': sorted(set(REQUIRED_COLUMNS) - set(columns)),
        'schema': tuple(columns),
    })

schema_check = pd.DataFrame(schema_rows)
print(f'고유 스키마 수: {schema_check.schema.nunique()}')
print(f'필수 컬럼 누락 파일: {schema_check.missing_required.astype(bool).sum()}')
display(schema_check.groupby('column_count').size().rename('file_count').to_frame())
display(schema_check.head())

## 샘플 파일 읽기

우선 첫 번째 파일 하나를 전체 로드한다. 종목 또는 날짜 선정 규칙은 아직 적용하지 않는다.

In [ ]:
SAMPLE_PATH = files[0]
sample_raw = pd.read_csv(SAMPLE_PATH, encoding='utf-8-sig')

print(f'파일: {SAMPLE_PATH}')
print(f'크기: {sample_raw.shape}')
display(sample_raw.head())
display(sample_raw.dtypes.rename('dtype').to_frame())

## 샘플 기본 품질 확인

확인용 복사본에서 시간과 OHLC dtype만 정리한다. 원본 파일은 수정하지 않는다.

In [ ]:
sample = sample_raw.copy()
sample['symbol'] = sample['symbol'].astype('string').str.strip().str.upper()
sample['timestamp_utc'] = pd.to_datetime(sample['timestamp_utc'], errors='coerce', utc=True)
for column in ['open', 'high', 'low', 'close']:
    sample[column] = pd.to_numeric(sample[column], errors='coerce')

required_missing = sample[REQUIRED_COLUMNS].isna().any(axis=1)
valid_ohlc = (
    sample[['open', 'high', 'low', 'close']].gt(0).all(axis=1)
    & sample['high'].ge(sample[['open', 'close']].max(axis=1))
    & sample['low'].le(sample[['open', 'close']].min(axis=1))
    & sample['high'].ge(sample['low'])
)
duplicate_key = sample.duplicated(['symbol', 'timestamp_utc'], keep=False)
ordered = sample.sort_values(['symbol', 'timestamp_utc'])
minute_delta = ordered.groupby('symbol')['timestamp_utc'].diff().dt.total_seconds().div(60)

quality_summary = pd.Series({
    'rows': len(sample),
    'symbols': sample.symbol.nunique(dropna=True),
    'start_utc': sample.timestamp_utc.min(),
    'end_utc': sample.timestamp_utc.max(),
    'required_missing_rows': int(required_missing.sum()),
    'invalid_ohlc_rows': int((~valid_ohlc).sum()),
    'duplicate_symbol_timestamp_rows': int(duplicate_key.sum()),
    'gaps_over_one_minute': int(minute_delta.gt(1).sum()),
}, name='value')
display(quality_summary.to_frame())
display(sample[REQUIRED_COLUMNS].head(10))

## 현재 단계 결론

이 노트북은 데이터를 읽고 구조를 확인하는 용도다. 다음 단계에서 전체 파일 품질 집계와 실제 정제 기준을 별도로 결정한다. 현재는 gap 제거, 결측 보간, feature 생성, 라벨 생성, 데이터 저장을 수행하지 않는다.